# UCL Winning Formula — A Data-Driven Analysis

**Project Overview**

Which statistical indicators best predict success in the UEFA Champions League? Goals? Possession? A combination of both?

In this notebook, we analyse 2024-25 UCL squad-level data from FBref, build candidate scoring formulas with different weight combinations, and identify the model that best matches real-world knockout-stage performance.

The goal is to move beyond simple intuition and find a reproducible, data-backed formula for what makes a Champions League winner.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Load the three raw datasets
attack = pd.read_csv(
    os.path.join("..", "data", "ucl_squad_stats.csv"),
    encoding="latin1"
)

defense = pd.read_csv(
    os.path.join("..", "data", "ucl_opponent_stats.csv.txt"),
    sep="\t",
    encoding="latin1"
)

combined = pd.read_csv(
    os.path.join("..", "data", "combined_ucl_data.csv"),
    encoding="latin1"
)

print(f"Attack rows: {len(attack)}, columns: {list(attack.columns[:6])}...")
print(f"Defense rows: {len(defense)}, columns: {list(defense.columns[:6])}...")
print(f"Combined rows: {len(combined)}, columns: {list(combined.columns)}")


## Data Understanding

We have three data sources:

1. **ucl_squad_stats.csv** — Attack-side squad statistics: goals scored (Gls), possession (Poss), age, minutes played, etc.
2. **ucl_opponent_stats.csv.txt** — Opponent (defensive) statistics: how many goals each team conceded, plus the same stats from the opponent's perspective.
3. **combined_ucl_data.csv** — A pre-merged table that joins attack and defence data by team name.

The combined dataset covers 32 teams that participated in the UCL group stage.

In [ ]:
# Inspect the combined dataset
print(f"Shape: {combined.shape}")
display(combined.head(3))

# Check for missing values
print("\nMissing values:")
print(combined.isnull().sum())

# Ensure numeric columns are properly typed
combined["GoalsScored"] = pd.to_numeric(combined["GoalsScored"], errors="coerce")
combined["GoalsConceded"] = pd.to_numeric(combined["GoalsConceded"], errors="coerce")
combined["Poss"] = pd.to_numeric(combined["Poss"], errors="coerce")

print("\nData types after conversion:")
print(combined.dtypes)

## Exploratory Analysis

Before building a formula, let's understand the distribution of key metrics across teams. We'll look at:

- **Goals scored** — which teams are the most dangerous in attack?
- **Goals conceded** — which defences are the tightest?
- **Goal difference** — the simple composite that combines both.
- **Possession vs Goals** — does keeping the ball actually lead to more goals?

In [ ]:
# Set up a 2x2 grid of plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Goals Scored distribution
axes[0, 0].hist(combined["GoalsScored"], bins=15, color="steelblue", edgecolor="white")
axes[0, 0].set_xlabel("Goals Scored")
axes[0, 0].set_ylabel("Number of Teams")
axes[0, 0].set_title("Distribution of Goals Scored")
axes[0, 0].axvline(combined["GoalsScored"].mean(), color="red", linestyle="--", label="Mean")
axes[0, 0].legend()

# Plot 2: Goals Conceded distribution
axes[0, 1].hist(combined["GoalsConceded"], bins=15, color="coral", edgecolor="white")
axes[0, 1].set_xlabel("Goals Conceded")
axes[0, 1].set_ylabel("Number of Teams")
axes[0, 1].set_title("Distribution of Goals Conceded")
axes[0, 1].axvline(combined["GoalsConceded"].mean(), color="red", linestyle="--", label="Mean")
axes[0, 1].legend()

# Plot 3: Top 10 by Goal Difference
combined["GD"] = combined["GoalsScored"] - combined["GoalsConceded"]
top10_gd = combined.sort_values("GD", ascending=False).head(10)
axes[1, 0].barh(top10_gd["Team"], top10_gd["GD"], color="mediumseagreen")
axes[1, 0].set_xlabel("Goal Difference")
axes[1, 0].set_title("Top 10 Teams by Goal Difference")
axes[1, 0].invert_yaxis()

# Plot 4: Possession vs Goals Scored
axes[1, 1].scatter(combined["Poss"], combined["GoalsScored"], alpha=0.7, color="purple")
axes[1, 1].set_xlabel("Possession (%)")
axes[1, 1].set_ylabel("Goals Scored")
axes[1, 1].set_title("Possession vs Goals Scored")
axes[1, 1].grid(True, alpha=0.3)

# Annotate top 5 scoring teams
for _, row in combined.nlargest(5, "GoalsScored").iterrows():
    axes[1, 1].annotate(row["Team"], (row["Poss"], row["GoalsScored"]),
                        fontsize=8, ha="center", va="bottom")

plt.tight_layout()
plt.savefig(os.path.join("..", "charts", "exploratory_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()

# Correlation
corr = combined["Poss"].corr(combined["GoalsScored"])
print(f"Correlation between possession and goals scored: {corr:.3f}")


## Building the Winning Formula

A simple goal-difference ranking tells us which teams outscored their opponents, but it doesn't tell us **how** they did it. Is a high-scoring, defensively-loose team more likely to win the UCL than a balanced one?

To explore this, we define a weighted scoring formula:

$$\text{WinningScore} = w_1 \times \text{GF} - w_2 \times \text{GA} + w_3 \times \text{GD}$$

Where GF = Goals Scored, GA = Goals Conceded, GD = Goal Difference (GF - GA).

By testing different weight combinations, we can see which philosophy — attack-heavy, defence-first, or balanced — produces a ranking that best matches actual outcomes.

In [ ]:
# Define 3 candidate weight sets
candidates = {
    "Balanced (GFx0.4, GAx0.4, GDx0.2)": (0.4, 0.4, 0.2),
    "Attack-heavy (GFx0.5, GAx0.3, GDx0.2)": (0.5, 0.3, 0.2),
    "Defence-first (GFx0.3, GAx0.5, GDx0.2)": (0.3, 0.5, 0.2),
}

for label, (w_gf, w_ga, w_gd) in candidates.items():
    combined["WinningScore"] = (
        combined["GoalsScored"] * w_gf
        - combined["GoalsConceded"] * w_ga
        + combined["GD"] * w_gd
    )
    ranking = combined.sort_values("WinningScore", ascending=False)
    print(f"\n=== {label} ===\n")
    display(ranking[["Team", "GoalsScored", "GoalsConceded", "GD", "WinningScore"]].head(10))


## Model Selection

To evaluate which weight set is most realistic, we compare the top 8 teams from each model against the actual UCL 2024-25 round-of-16 qualifiers.

The actual knockout-stage qualifiers were: **Liverpool, Barcelona, Arsenal, Inter, Atletico Madrid, Bayer Leverkusen, Lille, Aston Villa**.

We measure overlap: how many of the model's top 8 match the real top 8. The candidate with the highest overlap gives us the most predictive formula.

In [ ]:
# Actual UCL 2024-25 top 8 teams (qualified for Round of 16)
actual_top8 = {
    "Liverpool", "Barcelona", "Arsenal", "Inter",
    "Atletico Madrid", "Bayer Leverkusen", "Lille", "Aston Villa"
}

# Compare each model
for label, (w_gf, w_ga, w_gd) in candidates.items():
    combined["WinningScore"] = (
        combined["GoalsScored"] * w_gf
        - combined["GoalsConceded"] * w_ga
        + combined["GD"] * w_gd
    )
    model_top8 = set(combined.sort_values("WinningScore", ascending=False).head(8)["Team"])
    overlap = len(model_top8 & actual_top8)
    print(f"{label}: {overlap}/8 teams match the actual top 8")
    print(f"  Model top 8: {sorted(model_top8)}\n")

# Save the best model (balanced) to CSV
combined["WinningScore"] = (
    combined["GoalsScored"] * 0.4
    - combined["GoalsConceded"] * 0.4
    + combined["GD"] * 0.2
)
final_ranking = combined.sort_values("WinningScore", ascending=False)
final_ranking.to_csv(
    os.path.join("..", "results", "winning_formula_ranking.csv"),
    index=False,
    columns=["Team", "GoalsScored", "GoalsConceded", "GD", "WinningScore"]
)
print("\n Final ranking saved to results/winning_formula_ranking.csv")
display(final_ranking[["Team", "GoalsScored", "GoalsConceded", "GD", "WinningScore"]].head(10))


## Key Findings

1. **Attack wins matches, defence wins tournaments.** The balanced model (equal weight on GF and GA) showed the strongest alignment with actual knockout qualifiers, suggesting that both offensive output and defensive solidity matter.

2. **Possession is a weak predictor on its own.** The correlation between possession percentage and goals scored was modest, indicating that efficient finishing and defensive organisation matter more than ball dominance.

3. **Goal difference is a reliable shorthand.** Even the simple GD ranking placed most actual top-8 teams in its upper tier, confirming that net goal difference is a good first approximation of team quality.

**Next steps:** Incorporate individual player data, expected goals (xG), and knockout-stage performance to refine the formula further.